# Exercise 2 – Example: video → frames → timeline

This notebook shows the whole way from a video to a table of what is seen and said, with as little code as possible. **Run every cell and read the code.** Then open `video_extraction_tasks.ipynb`: there you build the same steps yourself for a new clip.

1. **Frames:** cut the clip into one picture per second.
2. **Describe:** let the OpenAI API describe every frame.
3. **Listen:** turn the speech into text with start and end times.
4. **Merge:** put what is seen and what is said on one timeline.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret. This notebook uses the **OpenAI API** (models `gpt-6-luna` and `whisper-1`); other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Python libraries.
import sys

import pandas as pd

# helper.py is in the main folder, one level up.
sys.path.append("..")
from helper import DATA, describe, extract_frames, show_images, transcribe, video_info

## 1. Frames

A video is a fast sequence of single pictures, called frames. `video_info()` shows how many frames per second (`fps`) and in total; `extract_frames()` keeps one per second and saves them in the folder `frames/`.

In [ ]:
video_path = DATA / "videos" / "ueli_scene3_talk.mp4"
video_info(video_path)

In [ ]:
paths = extract_frames(video_path)
show_images(paths)

## 2. Describe every frame

Try the prompt on one frame first, then describe all frames. `t` is the time in seconds.

In [ ]:
PROMPT = "Describe this video frame in one short sentence. Does Ueli look toward the camera?"
print(describe(PROMPT, paths[2]))

In [ ]:
rows = []
for t in range(len(paths)):
    rows.append({"t": t, "description": describe(PROMPT, paths[t])})

frames = pd.DataFrame(rows)
frames

## 3. Speech with start and end times

`transcribe()` sends the video to **Whisper** (model `whisper-1`), OpenAI's speech-to-text model: audio in, text out. It reads the sound track of the mp4 file itself.

The answer comes in **segments**: pieces of speech, each with a `start` and `end` time in seconds and its `text`. These times let us put what is said next to what is seen.

Whisper costs US$0.006 per audio minute, so this 8-second clip costs less than US$0.001. Swiss German often comes back as Standard German.

In [ ]:
speech = transcribe(video_path)
speech

## 4. Merge on time

A frame gets the text of a speech segment when `start ≤ t < end`. The time is the key that joins what is seen and what is said.

In [ ]:
frames["said"] = ""
for segment in speech.itertuples():
    inside = frames["t"].between(segment.start, segment.end, inclusive="left")
    frames.loc[inside, "said"] = segment.text

frames

Whisper decides how to split the speech: a short line can get its own segment or be merged into a longer one. A line shorter than the gap between two frames can fall between them – so keep the `speech` table next to the frames.

**Next:** open `video_extraction_tasks.ipynb` and build these steps yourself.

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')